In [12]:
import pandas as pd
from pathlib import Path

In [13]:
DATA_DIR = Path("../data/raw/mine/extended_streaming_history")

COLUMN_MAP = {          # basic-export names -> my canonical names
    "master_metadata_album_artist_name": "artist",
    "master_metadata_track_name": "track",
    "master_metadata_album_album_name": "album",
}

TIMEZONE = "America/Chicago"

files = sorted(DATA_DIR.glob("Streaming_History_Audio*.json"))
files

[PosixPath('../data/raw/mine/extended_streaming_history/Streaming_History_Audio_2021.json'),
 PosixPath('../data/raw/mine/extended_streaming_history/Streaming_History_Audio_2022.json'),
 PosixPath('../data/raw/mine/extended_streaming_history/Streaming_History_Audio_2023.json'),
 PosixPath('../data/raw/mine/extended_streaming_history/Streaming_History_Audio_2024.json'),
 PosixPath('../data/raw/mine/extended_streaming_history/Streaming_History_Audio_2025.json'),
 PosixPath('../data/raw/mine/extended_streaming_history/Streaming_History_Audio_2026.json')]

In [14]:
df = pd.concat([pd.read_json(f) for f in files], ignore_index=True)
df = df.rename(columns=COLUMN_MAP)

print("raw:", len(df))

df = df.drop(columns=["ip_addr"])                 # privacy: gone before anything else touches it

podcasts = df[df["track"].isna()].copy()          # podcasts + audiobooks have no track name
df = df[df["track"].notna()]
print("music only:", len(df), f"({len(podcasts)} podcast/audiobook rows set aside)")

df = df[df["ms_played"] > 0]
print("after zero-ms drop:", len(df))

bool_cols = ["shuffle", "skipped", "offline", "incognito_mode"]
df[bool_cols] = df[bool_cols].fillna(False).astype(bool)

df = df[~df["incognito_mode"]]
print("after incognito drop:", len(df))

raw: 61230
music only: 57922 (3308 podcast/audiobook rows set aside)
after zero-ms drop: 57203
after incognito drop: 57200


In [15]:
print(df.shape)
print(df.head())
print(df.dtypes)

(57200, 22)
                     ts                 platform  ms_played conn_country  \
0  2021-12-02T20:39:08Z  iOS 15.1.1 (iPhone14,2)     129941           US   
1  2021-12-02T20:39:10Z  iOS 15.1.1 (iPhone14,2)       1950           US   
2  2021-12-02T20:39:16Z  iOS 15.1.1 (iPhone14,2)       5340           US   
3  2021-12-02T20:39:39Z  iOS 15.1.1 (iPhone14,2)      22685           US   
4  2021-12-02T20:42:04Z  iOS 15.1.1 (iPhone14,2)     145244           US   

                                    track         artist  \
0              Killshot - Slowed + Reverb  Magdalena Bay   
1                                Killshot  Magdalena Bay   
2                     Secrets (Your Fire)  Magdalena Bay   
3                         Mercurial World  Magdalena Bay   
4  BIG PERSONA (feat. Tyler, The Creator)     Maxo Kream   

                                    album  \
0                                Killshot   
1    A Little Rhythm and a Wicked Feeling   
2                         Mercurial

In [16]:
df["ts"] = pd.to_datetime(df["ts"])
df['ts'] = pd.to_datetime(df['ts'], utc=True).dt.tz_convert(TIMEZONE)
df["ts"].min(), df["ts"].max()

(Timestamp('2021-12-02 14:39:08-0600', tz='America/Chicago'),
 Timestamp('2026-07-31 17:51:16-0500', tz='America/Chicago'))

In [17]:
print(df.isna().sum())
print(df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values("ts").head(10)

ts                             0
platform                       0
ms_played                      0
conn_country                   0
track                          0
artist                         0
album                          0
spotify_track_uri              0
episode_name               57200
episode_show_name          57200
spotify_episode_uri        57200
audiobook_title            57200
audiobook_uri              57200
audiobook_chapter_uri      57200
audiobook_chapter_title    57200
reason_start                   0
reason_end                     0
shuffle                        0
skipped                        0
offline                        0
offline_timestamp           5864
incognito_mode                 0
dtype: int64
29


,ts,platform,ms_played,conn_country,track,artist,album,spotify_track_uri,episode_name,episode_show_name,...,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
17,2021-12-03 04:10:18-06:00,web_player osx 10.15.7;chrome 95.0.4638.69;des...,963,US,Nasty Freestyle,T-Wayne,Nasty Freestyle,spotify:track:0PT7nlpo11hYYyfnBgtilT,NaN,NaN,...,NaN,NaN,NaN,remote,endplay,False,False,False,NaN,False
18,2021-12-03 04:10:18-06:00,web_player osx 10.15.7;chrome 95.0.4638.69;des...,963,US,Nasty Freestyle,T-Wayne,Nasty Freestyle,spotify:track:0PT7nlpo11hYYyfnBgtilT,NaN,NaN,...,NaN,NaN,NaN,remote,endplay,False,False,False,NaN,False
19,2021-12-03 04:11:35-06:00,web_player osx 10.15.7;chrome 95.0.4638.69;des...,1368,US,Wicked,Future,EVOL,spotify:track:6BbINUfGabVyiNFJpQXn3x,NaN,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,False,False,NaN,False
20,2021-12-03 04:11:35-06:00,web_player osx 10.15.7;chrome 95.0.4638.69;des...,1368,US,Wicked,Future,EVOL,spotify:track:6BbINUfGabVyiNFJpQXn3x,NaN,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,False,False,NaN,False
21,2021-12-03 04:11:59-06:00,web_player osx 10.15.7;chrome 95.0.4638.69;des...,6112,US,Back To Back,Drake,Back To Back,spotify:track:5lFDtgWsjRJu8fPOAyJIAK,NaN,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,False,False,NaN,False
22,2021-12-03 04:11:59-06:00,web_player osx 10.15.7;chrome 95.0.4638.69;des...,6112,US,Back To Back,Drake,Back To Back,spotify:track:5lFDtgWsjRJu8fPOAyJIAK,NaN,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,False,False,NaN,False
28,2021-12-03 04:15:25-06:00,web_player osx 10.15.7;chrome 95.0.4638.69;des...,5662,US,Know Yourself,Drake,If You're Reading This It's Too Late,spotify:track:5InOp6q2vvx0fShv3bzFLZ,NaN,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,False,False,NaN,False
29,2021-12-03 04:15:25-06:00,web_player osx 10.15.7;chrome 95.0.4638.69;des...,5662,US,Know Yourself,Drake,If You're Reading This It's Too Late,spotify:track:5InOp6q2vvx0fShv3bzFLZ,NaN,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,False,False,NaN,False
32,2021-12-03 04:24:53-06:00,web_player osx 10.15.7;chrome 95.0.4638.69;des...,838,US,"Crazy Brazy (feat. A$AP Rocky, A$AP Twelvyy & ...",A$AP Mob,Cozy Tapes: Vol. 1 Friends -,spotify:track:59e7E2LMPZ2bhW5G6aCwX8,NaN,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,False,False,NaN,False
33,2021-12-03 04:24:53-06:00,web_player osx 10.15.7;chrome 95.0.4638.69;des...,838,US,"Crazy Brazy (feat. A$AP Rocky, A$AP Twelvyy & ...",A$AP Mob,Cozy Tapes: Vol. 1 Friends -,spotify:track:59e7E2LMPZ2bhW5G6aCwX8,NaN,NaN,...,NaN,NaN,NaN,clickrow,endplay,False,False,False,NaN,False


In [18]:
print((df['ms_played']/60000).describe())
print((df['ms_played'] < 30_000).mean())  # 30k ms = 30 sec

count    57200.000000
mean         1.506978
std          1.595011
min          0.000017
25%          0.042496
50%          0.837008
75%          2.841321
max         20.259550
Name: ms_played, dtype: float64
0.46564685314685317


In [19]:
out = Path("../data/processed/streams_clean.parquet")
df.to_parquet(out)
print(out.exists(), pd.read_parquet(out).shape)

True (57200, 22)
